In [ ]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph,START,END

In [2]:

# Shared memory of the graph.
# Every node can:
# - Read state
# - Update state

class State(TypedDict):
    text: str
    word_count: int

In [3]:

def uppercase_node(state: State):
    return {
        "text": state["text"].upper()
    }

In [4]:
def count_words_node(state: State):
    return {
        "word_count": len(state["text"].split())
    }

In [5]:
graph_builder = StateGraph(State)

In [ ]:
## REGISTER NODES

In [7]:
graph_builder.add_node(
    "uppercase",
    uppercase_node
)

In [8]:
graph_builder.add_node(
    "count_words",
    count_words_node
)

In [ ]:
## Connect Nodes

In [9]:
graph_builder.add_edge(START,"uppercase")
graph_builder.add_edge("uppercase","count_words")
graph_builder.add_edge("count_words",END)

In [10]:
graph = graph_builder.compile()

In [11]:
result = graph.invoke(
        {
            "text": "hello langgraph"
        }
    )

In [12]:
print(result)

{'text': 'HELLO LANGGRAPH', 'word_count': 2}


State Overwrite

In [54]:
result = graph.invoke(
        {
            "text": "hello " # call with different msg
        }
    )

In [ ]:
print(result) # prev result is gone

{'text': 'HELLO ', 'word_count': 1}


In [32]:
## Hence we need reducers to store all the prev msgs4

# method 1 
import operator
class ReducerState1(TypedDict):
    items: Annotated[list,operator.add]
                            # red

In [43]:
# method 2 (preferred)
from langgraph.graph.message import add_messages
class ReducerState2(TypedDict):
    items: Annotated[list,add_messages]

In [44]:
## both works same